In [ ]:
# 공통 미션 완성 코드: B3(파이프라인) + B2(자재 선제 보충) + B1/M1(상자→창고→트럭 출하)

press3.speed = 'slow'
STEPS = ['LOAD', 'P1', 'P2', 'P3', 'INSPECT', 'SORT']
PRESS = {'P1': press1, 'P2': press2, 'P3': press3}

# ② 주문서를 납기 순으로 정렬하고, 생산할 (주문객체, 색상) 대기열(queue) 생성
order_list = sorted(orders, key=lambda o: o.due)
queue = []
for o in order_list:
    for color in ['red', 'blue', 'black']:
        queue += [(o, color)] * o.qty[color]

# 주문별 남은 생산 목표 수량 및 트럭 출하 완료 여부
todo = {o.id: sum(o.qty.values()) for o in order_list}
delivered = set()

def keep_material(color):
    """④ B2: 호퍼 잔량이 4개 이하이고 랙에 상자가 있으면 AGV-2로 미리 보충"""
    if feeder.level(color) <= 4 and material.crates(color) > 0:
        if supply_agv.busy_until <= clock.now:
            supply_agv.refill(color)

def ship_remaining_boxes():
    """⑤ B1: 상자에 들어있는 완성품을 모두 완성품 창고로 이동(AGV-3)"""
    for b in ['RED', 'BLUE', 'BLACK']:
        if boxes.count(b) > 0:
            agv.ship(b)

def deliver_ready_orders():
    """⑤ M1: 생산이 끝난 주문의 재고가 창고에 모이면 납품 트럭에 출하(AGV-4)"""
    for o in order_list:
        if o.id not in delivered and todo[o.id] == 0:
            if all(warehouse.stock[c] >= o.qty[c] for c in ['red', 'blue', 'black']):
                outbound_agv.deliver(o)
                delivered.add(o.id)
                print(f'주문 {o.id} 트럭 출하 · 시각 {clock.now:.1f}초 (납기 {o.due}초)')

def work(p):
    st = p.station
    if st == 'LOAD':
        o, color = p.plan
        keep_material(color)
        feeder.supply(color)
        scara.pick()
        scara.place(p)
    elif st in PRESS:
        PRESS[st].press()
    elif st == 'INSPECT':
        r = robot_a.inspect(p)
        if r == 'NG':
            # ③ 불량(NG) 발생 시 대기열 맨 앞에 즉시 재투입
            queue.insert(0, p.plan)
    elif st == 'SORT':
        o, _ = p.plan
        robot_b.pick(p)
        box = p.color if p.result == 'OK' else 'NG'
        if box != 'NG' and boxes.full(box):
            # ⑤ B1: 상자가 10자루 가득 차면 먼저 창고로 출발(AGV-3)
            agv.ship(box)
        robot_b.place(box)
        return_conv.send(p)

        if box != 'NG':
            todo[o.id] -= 1
            # 해당 주문의 마지막 펜이 상자에 들어가면 잔여 상자도 창고로 출하
            if todo[o.id] == 0:
                ship_remaining_boxes()
        return True
    conveyor.move(p, STEPS[STEPS.index(st) + 1])
    return False

feeder.on()
on_line = []

while queue or on_line:
    # 앞 파렛트부터 처리해야 앞 역이 비어 뒤 파렛트가 전진함
    for p in list(on_line):
        if work(p):
            on_line.remove(p)

    # ① B3: 대기 파렛트가 있고 LOAD 역이 비어 있으면 즉시 중첩 투입 (최대 5장 동시 가동)
    if queue and stocker.count > 0 and conveyor.at('LOAD') is None:
        p = stocker.release()
        p.plan = queue.pop(0)
        on_line.append(p)

    deliver_ready_orders()

feeder.off()
ship_remaining_boxes()

# 마지막 주문 상자가 AGV-3로 창고에 도착할 때까지 대기 후 최종 트럭 출하
while len(delivered) < len(order_list) and clock.now < 400:
    deliver_ready_orders()
    if len(delivered) == len(order_list):
        break
    # 남은 1~2초를 흘려보내기 위해 빈 컨베이어 대기
    conveyor.wait(1) if hasattr(conveyor, 'wait') else None
    if not hasattr(conveyor, 'wait'):
        break

print('전체 종료 %.1f초 / 창고 %s' % (clock.now, warehouse.stock))